# Paper9 自然资源部完整流程与日志

在 Notebook 中运行完整 Paper9 工作流：配置校验、预演（dry-run）命令核查、可选的准备/采样/训练/规划（prepare/sample/train/plan）执行、成果审计、运行清单（manifest）查看和阶段日志检查。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import display

from paper9_mnr.notebook_utils import latest_run_manifest, manifest_log_entries, manifest_stage_table, project_root

ROOT = project_root()
CONFIG = os.environ.get("PAPER9_CONFIG", "configs/real_data_from_authority_slope.yml")
LOG_DIR = "outputs/logs"

print(f"ROOT={ROOT}")
print(f"CONFIG={CONFIG}")
print(f"LOG_DIR={LOG_DIR}")

def run_command(args, check=False):
    result = subprocess.run(args, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"returncode={result.returncode}")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}")
    return result

## 1. 校验配置

In [ ]:
check_config = run_command([sys.executable, "-m", "paper9_mnr.cli", "check-config", CONFIG], check=True)

## 2. 预演（dry-run）查看完整流程命令

In [ ]:
dry_run = run_command([
    sys.executable,
    "scripts/run_full_pipeline.py",
    CONFIG,
    "--dry-run",
    "--log-dir",
    LOG_DIR,
], check=True)

## 3. 执行准备/采样/训练/规划（prepare -> sample -> train -> plan）

只有在预演（dry-run）命令和配置确认无误后，才把 `RUN_PIPELINE = True`。该操作会执行完整工作流，在自然资源部真实数据上可能需要数分钟到数小时。

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    full_run = run_command([
        sys.executable,
        "scripts/run_full_pipeline.py",
        CONFIG,
        "--log-dir",
        LOG_DIR,
    ], check=True)
else:
    print("RUN_PIPELINE is False. Change it to True to execute prepare, sample, train, and plan from Notebook.")

## 4. 审计输出成果

In [ ]:
RUN_AUDIT = True

if RUN_AUDIT:
    audit = run_command([sys.executable, "scripts/05_audit.py", CONFIG, "--write"], check=True)
else:
    print("RUN_AUDIT is False.")

## 5. 查看最近一次运行清单（manifest）

In [ ]:
manifest = latest_run_manifest(LOG_DIR)
display(manifest)

if manifest:
    print(f"dry_run={manifest.get('dry_run')} status={manifest.get('status')}")
    stages = manifest_stage_table(manifest)
    if not stages.empty:
        display(stages)
else:
    print("No run manifest found yet.")

## 6. 查看阶段日志末尾

In [ ]:
TAIL_LINES = 80

entries = manifest_log_entries(manifest, root=ROOT) if manifest else []

if entries:
    for entry in entries:
        resolved = entry["log_path"]
        print("\n" + "=" * 80)
        print(f"{entry.get('stage')} log: {resolved}")
        if entry["exists"]:
            lines = resolved.read_text(encoding="utf-8", errors="replace").splitlines()
            print("\n".join(lines[-TAIL_LINES:]))
        else:
            print("Log file does not exist.")
else:
    print("No per-stage log files are listed in this manifest. Dry-run manifests only contain command previews.")